In [7]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
from pathlib import Path
import sys


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project"
)

CUSTOM_SRC = (
    PROJECT_ROOT
    / "notebooks"
    / "generative"
    / "custom"
    / "src"
)

assert CUSTOM_SRC.is_dir(), (
    f"Custom source directory was not found:\n"
    f"{CUSTOM_SRC}"
)

if str(CUSTOM_SRC) not in sys.path:
    sys.path.insert(
        0,
        str(CUSTOM_SRC),
    )

print("Custom source directory:")
print(CUSTOM_SRC)

Custom source directory:
/content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/notebooks/generative/custom/src


In [9]:
from utils.model.custom_unet_2d import CustomUNet2DModel

print("CustomUNet2DModel import passed.")

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


CustomUNet2DModel import passed.


In [10]:
from base import config, model, noise_scheduler

print("base.py import passed.")

base.py import passed.


In [11]:
print("Run ID:", config.run_id)
print("Results directory:", config.results_dir)
print("Training timesteps:", config.num_train_timesteps)

print("\nModel sample size:", model.config.sample_size)
print("Model input channels:", model.config.in_channels)
print("Model output channels:", model.config.out_channels)
print("Layers per block:", model.config.layers_per_block)
print("Block channels:", model.config.block_out_channels)

print(
    "\nScheduler timesteps:",
    noise_scheduler.config.num_train_timesteps,
)

Run ID: 1
Results directory: /content/drive/MyDrive/NMA-DL2026/NMA_BrainMRI_Project/checkpoints/custom/runs/01
Training timesteps: 2000

Model sample size: (160, 224)
Model input channels: 1
Model output channels: 1
Layers per block: 4
Block channels: (32, 32, 64, 64, 128, 128)

Scheduler timesteps: 2000


In [12]:
assert tuple(model.config.sample_size) == (
    160,
    224,
)

assert model.config.in_channels == 1
assert model.config.out_channels == 1
assert model.config.layers_per_block == 4

assert tuple(
    model.config.block_out_channels
) == (
    32,
    32,
    64,
    64,
    128,
    128,
)

assert (
    noise_scheduler.config.num_train_timesteps
    == 2000
)

print("Model and scheduler configuration passed.")

Model and scheduler configuration passed.


In [13]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

assert total_parameters > 0
assert trainable_parameters == total_parameters

Total parameters: 46,055,937
Trainable parameters: 46,055,937


In [14]:
import torch


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cuda


In [15]:
batch_size = 1

clean_sample = torch.randn(
    batch_size,
    1,
    160,
    224,
    device=device,
)

clean_control = torch.zeros(
    batch_size,
    1,
    160,
    224,
    device=device,
)

noise = torch.randn_like(
    clean_sample
)

timesteps = torch.tensor(
    [1000],
    dtype=torch.long,
    device=device,
)

In [16]:
noisy_sample = noise_scheduler.add_noise(
    original_samples=clean_sample,
    noise=noise,
    timesteps=timesteps,
)

noisy_control = noise_scheduler.add_noise(
    original_samples=clean_control,
    noise=noise,
    timesteps=timesteps,
)

print("Clean sample:", clean_sample.shape)
print("Noisy sample:", noisy_sample.shape)
print("Noisy control:", noisy_control.shape)

Clean sample: torch.Size([1, 1, 160, 224])
Noisy sample: torch.Size([1, 1, 160, 224])
Noisy control: torch.Size([1, 1, 160, 224])


In [17]:
model = model.to(device)
model.eval()

with torch.inference_mode():
    if device.type == "cuda":
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        ):
            output = model(
                sample=noisy_sample,
                control=noisy_control,
                timestep=timesteps,
            ).sample
    else:
        output = model(
            sample=noisy_sample,
            control=noisy_control,
            timestep=timesteps,
        ).sample

print("Model output shape:", output.shape)
print("Model output dtype:", output.dtype)
print(
    "Output contains finite values:",
    bool(torch.isfinite(output).all()),
)

Model output shape: torch.Size([1, 1, 160, 224])
Model output dtype: torch.float16
Output contains finite values: True


In [18]:
assert output.shape == (
    batch_size,
    1,
    160,
    224,
)

assert torch.isfinite(
    output
).all()

print(
    "Custom U-Net forward-pass test passed."
)

Custom U-Net forward-pass test passed.


In [19]:
model = model.cpu()

del clean_sample
del clean_control
del noise
del noisy_sample
del noisy_control
del output

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Temporary GPU memory released.")

Temporary GPU memory released.
